# 06 Rag Evaluation

Companion notebook to the textbook. Run from the `notebooks/` folder inside the project (the code uses `..` paths). Cells marked NEEDS INTERNET download a Hugging Face model on first run.

# Chapter 17: Retrieval-Augmented Generation (RAG)

## 17.1 What RAG is and why it exists

## 17.2 The complete workflow and architecture

## 17.3 Running the full pipeline end to end

In [ ]:
import sys, tempfile
sys.path.insert(0, "..")
from pathlib import Path
from src.embeddings import HashingEmbedder
from src.llm import ExtractiveLLM
from src.rag_pipeline import RAGPipeline
from src.utils import load_config
cfg = load_config(); cfg["embeddings"]["backend"] = "hashing"
pipe = RAGPipeline(cfg, embedder=HashingEmbedder(), llm=ExtractiveLLM(), persist_dir=Path(tempfile.mkdtemp()) / "db")
for pdf in sorted(Path("../data/sample_pdfs").glob("*.pdf")):
    info = pipe.ingest(pdf)
    print("ingested", info["source"], "-", info["added_chunks"], "chunks")
for q in ["How does RAG reduce hallucination?", "What is the default rate limit of the Notely API?",
          "How do I bake sourdough bread?"]:
    r = pipe.ask(q)
    print("\nQ:", q)
    print("A:", r["answer"])
    for s in r["sources"]:
        print(f"   [{s['n']}] {s['source']} page {s['page']} (score {s['score']})")
    print("   supported:", r["supported"])

## 17.4 Handling insufficient evidence

## 17.5 Exercises

## 17.6 Interview questions and answers

# Chapter 20: RAG Evaluation

## 20.1 Why evaluate

## 20.2 The evaluation dataset

## 20.3 Evaluating the actual application: retrieval and citations

In [ ]:
import sys, json
sys.path.insert(0, "..")
from evaluation.evaluate_retrieval import evaluate
from evaluation.evaluate_answers import run
print(json.dumps(evaluate("hashing", "recursive", 800, 120, 4)))
print(json.dumps(run("hashing", "extractive")))

## 20.4 Comparing configurations fairly

In [ ]:
import sys
sys.path.insert(0, "..")
from evaluation.evaluate_retrieval import evaluate
rows = [evaluate("hashing", "recursive", size, size // 6, k) for size in (200, 400, 800) for k in (1, 4)]
for r in rows:
    k = r["top_k"]
    print(f"size={r['chunk_size']:3d} top_k={k}  hit={r[f'hit@{k}']}  precision={r[f'precision@{k}']}  MRR={r['MRR']}")

## 20.5 Faithfulness, RAGAS and manual evaluation

## 20.6 Hallucination evaluation

## 20.7 Interview questions and answers

# Chapter 22: Testing and Debugging

## 22.1 Why test an AI application

## 22.2 What each test file covers

## 22.3 Running the whole suite

In [ ]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True, cwd="..")
print(r.stdout.strip().splitlines()[-1])

## 22.4 Writing a test: citations and unsupported questions

In [ ]:
import inspect, sys
sys.path.insert(0, "..")
src = open("../tests/test_rag_pipeline.py").read().split("\n\n\n")
print(src[1]); print(); print(src[2])

## 22.5 Debugging common failures

## 22.6 Interview questions and answers